# 4일차 실습: 실시간 객체 검출 앱 (OpenVINO)
---
## 오늘 목표

OpenVINO 사전 학습 모델로 **실시간 웹캠에서 얼굴을 검출**하는 앱을 완성합니다.

### 오늘 흐름
1. OpenVINO 모델 클래스화
2. 실시간 얼굴 검출 앱
3. 모자이크/블러 효과 추가
4. 미니 프로젝트: 자유 주제 앱

## 0. Colab 환경 설정

> **Colab에서 실행하는 경우** 아래 셀을 먼저 실행하세요.  
> 로컬에서 실행하는 경우 이 섹션은 건너뛰어도 됩니다.

In [2]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('OpenVINO 설치 중...')
    os.system('pip install -q openvino openvino-dev')

    from google.colab import drive
    drive.mount('/content/drive')

    MODEL_DIR = '/content/drive/MyDrive/CV_course/models'
    os.makedirs(MODEL_DIR, exist_ok=True)

    face_path = f'{MODEL_DIR}/intel/face-detection-adas-0001/FP32/face-detection-adas-0001.xml'
    person_path = f'{MODEL_DIR}/intel/person-detection-retail-0013/FP32/person-detection-retail-0013.xml'

    if not os.path.exists(face_path):
        print('얼굴 검출 모델 다운로드 중... (최초 1회)')
        os.system(f'omz_downloader --name face-detection-adas-0001 --output_dir {MODEL_DIR}')
    else:
        print('얼굴 검출 모델: Drive에서 로드')

    if not os.path.exists(person_path):
        print('사람 검출 모델 다운로드 중... (최초 1회)')
        os.system(f'omz_downloader --name person-detection-retail-0013 --output_dir {MODEL_DIR}')
    else:
        print('사람 검출 모델: Drive에서 로드')

    # 테스트 이미지
    os.makedirs('images', exist_ok=True)
    if not os.path.exists('images/people.jpg'):
        os.system('wget -q -O images/people.jpg https://upload.wikimedia.org/wikipedia/commons/thumb/b/b6/Image_created_with_a_mobile_phone.png/320px-Image_created_with_a_mobile_phone.png')

    print('Colab 환경 설정 완료!')

    # 웹캠 실시간 앱은 Colab에서 동작하지 않아요
    # 섹션 3, 4 (실시간 앱) 대신 이미지 파일로 테스트합니다
    print('주의: 실시간 웹캠 섹션(3, 4)은 Colab 미지원 → 이미지 파일 테스트로 대체')

else:
    MODEL_DIR = r'C:\Users\ComHolic\Desktop\AI_Folder\AI\04. Coding\VSCode\cv\models'
    print('로컬 환경 — 기존 모델 경로 사용')


로컬 환경 — 기존 모델 경로 사용


## 1. 준비

아래는 **선생님이 미리 실행**합니다.

In [2]:
import sys
print(sys.version)
import numpy
print(numpy.__version__)
import cv2
import numpy as np

from openvino.runtime import Core
print("hello")

try:
    from openvino.runtime import Core
    ie = Core()
    print("OpenVINO Core 로드 성공!")
    
    # 내 컴퓨터에 있는 실제 모델 파일 경로를 여기에 넣으세요

    # model = ie.read_model(model="모델파일명.xml") 
except Exception as e:
    print(f"에러 발생: {e}")

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
1.26.4
hello
OpenVINO Core 로드 성공!


In [5]:
try:
    ie = Core()
    # 'CPU' 혹은 'AUTO'를 주로 사용합니다.
    MODEL_DIR = r'C:\Users\ComHolic\Desktop\AI_Folder\AI\04. Coding\VSCode\cv\models'

    compiled_model = ie.compile_model(model=f'{MODEL_DIR}\\intel\\face-detection-adas-0001\\FP32\\face-detection-adas-0001.xml', device_name="CPU") 
    print("모델 컴파일 성공!")
except Exception as e:
    print(f"오류 내용: {e}")

모델 컴파일 성공!


In [9]:
#import cv2
#import numpy as np
#from openvino.runtime import Core
import time
from datetime import datetime
# ie = Core()  # 이 경우 'Core'가 아닌 'ov.Core()'로 사용

#print('사용 가능한 디바이스:', ie.available_devices)
print('\n필요 모델 다운로드 명령어:')
print('  omz_downloader --name face-detection-adas-0001')
print('  omz_downloader --name person-detection-retail-0013')


필요 모델 다운로드 명령어:
  omz_downloader --name face-detection-adas-0001
  omz_downloader --name person-detection-retail-0013


## 2. OpenVINO 모델 래퍼 클래스

매번 같은 전처리/후처리 코드를 반복하지 않도록 클래스로 묶습니다.

In [6]:
class OpenVINODetector:
    def __init__(self, model_path, device='CPU', confidence=0.5):
        ie = Core()
        model = ie.read_model(model_path)
        self.compiled = ie.compile_model(model, device)
        self.input_layer  = self.compiled.input(0)
        self.output_layer = self.compiled.output(0)
        self.confidence = confidence
        self.input_h = self.input_layer.shape[2]
        self.input_w = self.input_layer.shape[3]
        print(f'모델 로드 완료! 입력: {self.input_w}x{self.input_h}')

    def preprocess(self, frame):
        img = cv2.resize(frame, (self.input_w, self.input_h))
        img = img.transpose(2, 0, 1)          # HWC -> CHW
        return np.expand_dims(img, 0).astype(np.float32)

    def detect(self, frame):
        h, w = frame.shape[:2]
        results = self.compiled([self.preprocess(frame)])[self.output_layer]
        detections = []
        for det in results[0][0]:
            conf = det[2]
            if conf > self.confidence:
                x1 = max(0, int(det[3] * w))
                y1 = max(0, int(det[4] * h))
                x2 = min(w, int(det[5] * w))
                y2 = min(h, int(det[6] * h))
                detections.append((x1, y1, x2, y2, float(conf)))
        return detections

print('OpenVINODetector 클래스 정의 완료!')

OpenVINODetector 클래스 정의 완료!


## 3. 실시간 얼굴 검출 앱

In [10]:
MODEL_DIR = r'C:\Users\ComHolic\Desktop\AI_Folder\AI\04. Coding\VSCode\cv\models'

# 얼굴 검출 모델 로드
face_detector = OpenVINODetector(
    f'{MODEL_DIR}\\intel\\face-detection-adas-0001\\FP32\\face-detection-adas-0001.xml',
    confidence=0.5
)

camera = cv2.VideoCapture(0)
frame_times = []

print('앱 시작! q: 종료')
while True:
    t_start = time.time()
    ret, frame = camera.read()
    if not ret:
        break

    detections = face_detector.detect(frame)
    result = frame.copy()

    for (x1, y1, x2, y2, conf) in detections:
        cv2.rectangle(result, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(result, f'{conf:.0%}', (x1, y1-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    frame_times.append(time.time() - t_start)
    if len(frame_times) > 30:
        frame_times.pop(0)
    fps = 1.0 / (sum(frame_times) / len(frame_times))

    cv2.putText(result, f'FPS: {fps:.1f}', (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(result, f'얼굴: {len(detections)}명', (10, 65),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

    cv2.imshow('얼굴 검출 (q: 종료)', result)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

camera.release()
cv2.destroyAllWindows()

모델 로드 완료! 입력: 672x384
앱 시작! q: 종료


## 4. 얼굴 모자이크 앱

검출된 얼굴 영역에 자동으로 모자이크를 적용합니다.

In [11]:
def apply_mosaic(frame, x1, y1, x2, y2, block_size=15):
    roi = frame[y1:y2, x1:x2]
    if roi.size == 0:
        return frame
    h, w = roi.shape[:2]
    small  = cv2.resize(roi, (w // block_size, h // block_size))
    mosaic = cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)
    frame[y1:y2, x1:x2] = mosaic
    return frame

camera = cv2.VideoCapture(0)
mosaic_on = True

print('앱 시작! m: 모자이크/블러 전환  q: 종료')
while True:
    ret, frame = camera.read()
    if not ret:
        break

    result = frame.copy()
    detections = face_detector.detect(frame)

    for (x1, y1, x2, y2, conf) in detections:
        if mosaic_on:
            result = apply_mosaic(result, x1, y1, x2, y2, block_size=15)
        else:
            roi = result[y1:y2, x1:x2]
            if roi.size > 0:
                result[y1:y2, x1:x2] = cv2.GaussianBlur(roi, (51, 51), 0)

    mode_text = '모자이크' if mosaic_on else '블러'
    cv2.putText(result, f'모드: {mode_text} (m:전환)', (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(result, f'얼굴: {len(detections)}명', (10, 65),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

    cv2.imshow('얼굴 모자이크 (m:모드전환  q:종료)', result)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('m'):
        mosaic_on = not mosaic_on

camera.release()
cv2.destroyAllWindows()

앱 시작! m: 모자이크/블러 전환  q: 종료


## 🎯 도전 과제 — GenAI 활용: 기능 추가

원하는 기능을 GenAI로 구현해보세요!

---
**프롬프트 A: 검출 카운터 추가**
```
OpenCV 실시간 얼굴 검출 앱에 다음 기능을 추가해줘:
- 화면 오른쪽 상단에 '총 검출 횟수: N회' 표시
- 얼굴이 처음 검출될 때마다 카운터 증가
- 'r' 키를 누르면 카운터 리셋
```

**프롬프트 B: 검출 얼굴 저장**
```
OpenCV 실시간 얼굴 검출 앱에서
's' 키를 누르면 현재 검출된 얼굴 영역만 crop해서
'face_001.png', 'face_002.png' 형태로 저장하는 기능을 추가해줘.
```

**프롬프트 C: Edge + 검출 결합**
```
OpenCV 실시간 앱에서 'e' 키를 누르면
현재 프레임에 Canny 엣지를 배경으로 깔고
그 위에 얼굴 검출 바운딩 박스를 표시하는 모드를 추가해줘.
```
---

In [17]:
# 여기에 GenAI가 생성한 코드를 붙여넣으세요
# your code here
import cv2
import time
import os
import numpy as np

# ... (OpenVINODetector 클래스 정의가 상단에 있다고 가정) ...

MODEL_DIR = r'C:\Users\ComHolic\Desktop\AI_Folder\AI\04. Coding\VSCode\cv\models'

# 얼굴 검출 모델 로드
face_detector = OpenVINODetector(
    f'{MODEL_DIR}\\intel\\face-detection-adas-0001\\FP32\\face-detection-adas-0001.xml',
    confidence=0.5
)
# --- 추가된 설정: 저장 폴더 생성 ---
save_dir = 'captured_faces'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

face_save_count = 0  # 저장 파일 번호 카운터
# -------------------------------

# --- 추가된 변수 ---
total_detected_count = 0  # 총 검출 횟수 카운터
was_face_present = False  # 이전 프레임에 얼굴이 있었는지 여부

# --- 1. 설정 및 상태 변수 추가 ---
is_edge_mode = False  # 엣지 모드 활성화 여부
save_dir = 'captured_faces'
if not os.path.exists(save_dir): os.makedirs(save_dir)
face_save_count = 0
total_detected_count = 0
was_face_present = False

# ------------------
camera = cv2.VideoCapture(0)
frame_times = []

print('앱 시작! q: 종료, r: 카운트 리셋')

while True:
    t_start = time.time()
    ret, frame = camera.read()
    if not ret:
        break

    detections = face_detector.detect(frame)
    # --- 2. 배경 처리 로직 (Edge 모드 판별) ---
    if is_edge_mode:
        # Canny 엣지 검출 (임계값 100, 200)
        edges = cv2.Canny(frame, 100, 200)
        # 흑백 엣지 지도를 3채널 BGR로 변환 (박스 색상을 입히기 위함)
        result = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
    else:
        result = frame.copy()

    # --- 카운팅 로직 추가 ---
    # 현재 프레임에 얼굴이 있고, 이전 프레임에는 없었을 때만 카운트 증가
    is_face_present_now = len(detections) > 0
    if is_face_present_now and not was_face_present:
        total_detected_count += 1
    
    # 현재 상태를 다음 프레임을 위해 저장
    was_face_present = is_face_present_now
    # -----------------------

    # 얼굴 박스 및 개별 신뢰도 표시
    for (x1, y1, x2, y2, conf) in detections:
        cv2.rectangle(result, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(result, f'{conf:.0%}', (x1, y1-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # UI 정보 표시 (생략 가능하나 가시성을 위해 유지)
    mode_text = "MODE: EDGE" if is_edge_mode else "MODE: NORMAL"
    cv2.putText(result, mode_text, (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 255), 2)
    
    # FPS 계산
    frame_times.append(time.time() - t_start)
    if len(frame_times) > 30:
        frame_times.pop(0)
    fps = 1.0 / (sum(frame_times) / len(frame_times))

    # 왼쪽 상단 정보 표시
    cv2.putText(result, f'FPS: {fps:.1f}', (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(result, f'현재 얼굴: {len(detections)}명', (10, 65),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

    # --- 오른쪽 상단 총 검출 횟수 표시 추가 ---
    count_text = f'총 검출 횟수: {total_detected_count}회'
    # 텍스트 크기를 계산하여 오른쪽 끝에 맞춤
    text_size = cv2.getTextSize(count_text, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)[0]
    text_x = result.shape[1] - text_size[0] - 10
    cv2.putText(result, count_text, (text_x, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 255), 2)
    # ---------------------------------------

    cv2.imshow('얼굴 검출 (q: 종료, r: 리셋)', result)
    
    key = cv2.waitKey(1) & 0xFF
    # 'q' 키를 누르면 종료
    if key == ord('q'):
        break
    # --- 'r' 키를 누르면 카운터 리셋 추가 ---
    elif key == ord('r'):
        total_detected_count = 0
        print('카운터가 리셋되었습니다.')
    # --------------------------------------
    # --- 's' 키: 얼굴 크롭 및 저장 기능 추가 ---
    elif key == ord('s'):
        if len(detections) > 0:
            for i, (x1, y1, x2, y2, conf) in enumerate(detections):
                face_save_count += 1
                
                # 1. 얼굴 영역 크롭 (좌표 유효성 체크 포함)
                # 이미지가 화면 밖으로 나가는 경우를 대비해 0과 최대치로 제한합니다.
                h, w, _ = frame.shape
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w, x2), min(h, y2)
                
                cropped_face = frame[y1:y2, x1:x2]
                
                # 2. 파일명 형식 생성 (face_001.png 등)
                file_name = f'face_{face_save_count:03d}.png'
                file_path = os.path.join(save_dir, file_name)
                
                # 3. 이미지 저장
                cv2.imwrite(file_path, cropped_face)
                print(f'저장 완료: {file_path}')
        else:
            print("저장할 얼굴이 검출되지 않았습니다.")
    # ------------------------------------------
    # --- 4. 'e' 키: 엣지 모드 토글 기능 추가 ---
    elif key == ord('e'):
        is_edge_mode = not is_edge_mode
        print(f"엣지 모드 {'활성화' if is_edge_mode else '비활성화'}")
        
camera.release()
cv2.destroyAllWindows()

모델 로드 완료! 입력: 672x384
앱 시작! q: 종료, r: 카운트 리셋
엣지 모드 활성화
엣지 모드 비활성화
엣지 모드 활성화
엣지 모드 비활성화
엣지 모드 활성화
엣지 모드 비활성화
엣지 모드 활성화
엣지 모드 비활성화


### 배경 블러

In [ ]:
# 모델 파일 경로 (실제 경로로 수정 필요)
MODEL_DIR = r'C:\Users\ComHolic\Desktop\AI_Folder\AI\04. Coding\VSCode\cv\models'

# 얼굴 검출 모델 로드
face_detector = OpenVINODetector(
    f'{MODEL_DIR}\\intel\\face-detection-adas-0001\\FP32\\face-detection-adas-0001.xml',
    confidence=0.5
)

camera = cv2.VideoCapture(0)
blur_on = True

print("앱 시작! [b: 블러 토글] [q: 종료]")

while True:
    ret, frame = camera.read()
    if not ret: break
    
    detections = face_detector.detect(frame)
    
    if blur_on:
        # 1. 전체 화면을 강하게 블러 처리 (배경 생성)
        display_frame = cv2.GaussianBlur(frame, (99, 99), 0)
        
        for (x1, y1, x2, y2, conf) in detections:
            # 2. 검출된 얼굴 영역(ROI)만 원본에서 복사
            face_roi = frame[y1:y2, x1:x2]
            
            # 3. 블러된 배경 위에 원본 얼굴 영역을 덮어씌움 (복원)
            if face_roi.size > 0:
                display_frame[y1:y2, x1:x2] = face_roi
                
            # 시각적인 피드백을 위해 얇은 테두리 추가
            cv2.rectangle(display_frame, (x1, y1), (x2, y2), (0, 255, 0), 1)
    else:
        display_frame = frame.copy()

    # 상태 표시
    status_text = f"Privacy Blur: {'ON' if blur_on else 'OFF'} (Press 'b' to toggle)"
    cv2.putText(display_frame, status_text, (10, 30), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    cv2.imshow('Face Privacy Filter', display_frame)
    
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('b'):
        blur_on = not blur_on

camera.release()
cv2.destroyAllWindows()

---
## 오전 심화 — 웹캠 실시간 앱 3종

Day3 파이프라인을 기반으로 3가지 앱을 추가로 만듭니다.

| 앱 | 핵심 기술 |
|---|---|
| 4-1. 모자이크 + 감정 동시 | 얼굴 검출 + 모자이크 + 감정 오버레이 |
| 4-2. 움직임 감지 | 프레임 차이 + Contour |
| 4-3. 출석 체크 | 얼굴 검출 + 카운터 + 시간 측정 |

### 4-1. 모자이크 + 감정 동시 앱

얼굴을 검출하고 `m` 키로 모자이크/감정 모드를 전환합니다.

```
모드 1 (기본): 얼굴 바운딩 박스 + 감정 텍스트
모드 2 (m키):  얼굴 영역 모자이크 처리
```

In [ ]:
from PIL import ImageFont, ImageDraw, Image

font_path   = 'C:/Windows/Fonts/malgun.ttf'
font_medium = ImageFont.truetype(font_path, 24)
font_small  = ImageFont.truetype(font_path, 18)

# 감정 인식 추가
emotion_compiled = ie.compile_model(
    ie.read_model(f'{MODEL_DIR}/intel/emotions-recognition-retail-0003/FP32/emotions-recognition-retail-0003.xml'),
    'CPU'
)

def put_korean_text(img, text, position, color=(0,255,255), font=None):
    if font is None: font = font_medium
    img_pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    draw    = ImageDraw.Draw(img_pil)
    draw.text(position, text, font=font, fill=(color[2], color[1], color[0]))
    return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)

def apply_mosaic(frame, x1, y1, x2, y2, block_size=15):
    roi = frame[y1:y2, x1:x2]
    if roi.size == 0: return frame
    h, w   = roi.shape[:2]
    small  = cv2.resize(roi, (max(1,w//block_size), max(1,h//block_size)))
    mosaic = cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)
    frame[y1:y2, x1:x2] = mosaic
    return frame

EMOTIONS      = ['neutral', 'happy', 'sad', 'surprise', 'anger']
EMOTION_COLOR = {'happy':(0,255,0),'neutral':(255,255,0),
                 'sad':(255,100,0),'surprise':(0,200,255),'anger':(0,0,255)}

camera     = cv2.VideoCapture(0)
mosaic_on  = False

print('앱 시작!  m: 모자이크/감정 전환  q: 종료')

while True:
    ret, frame = camera.read()
    if not ret: break
    result = frame.copy()
    h, w   = frame.shape[:2]

    # 얼굴 검출
    detections = face_detector.detect(frame)

    for (x1, y1, x2, y2, conf) in detections:
        if conf > 0.3:
            if mosaic_on:
                result = apply_mosaic(result, x1, y1, x2, y2)
            else:
                roi = frame[y1:y2, x1:x2]
                if roi.size > 0:
                    e_in  = np.expand_dims(cv2.resize(roi,(64,64)).transpose(2,0,1), 0).astype(np.float32)
                    probs = emotion_compiled([e_in])[emotion_compiled.output(0)][0].flatten()
                    emo   = EMOTIONS[int(np.argmax(probs))]
                    color = EMOTION_COLOR.get(emo, (255,255,255))
                    cv2.rectangle(result, (x1,y1), (x2,y2), color, 2)
                    result = put_korean_text(result, f'{emo} {probs.max():.0%}',
                                             (x1, max(0,y1-30)), color)

    mode_text = '모드: 모자이크' if mosaic_on else '모드: 감정 인식'
    result = put_korean_text(result, f'{mode_text}  (m: 전환)', (10, 10),
                         (0, 255, 255), font_medium)

    cv2.imshow('모자이크+감정 앱 (m:전환  q:종료)', result)
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'): break
    elif key == ord('m'):
        mosaic_on = not mosaic_on
        print('모자이크 ON' if mosaic_on else '감정 인식 ON')

camera.release()
cv2.destroyAllWindows()


### 4-2. 움직임 감지 앱

이전 프레임과 현재 프레임의 차이로 움직임을 감지합니다.

```
이전 프레임 - 현재 프레임 → 차이 이미지
    → 임계값 적용 → Contour 검출 → 움직임 영역 표시
```

> 💡 CCTV, 보안 카메라의 움직임 감지가 이 원리로 동작해요!

In [ ]:
camera    = cv2.VideoCapture(0)
prev_gray = None
threshold = 25    # 움직임 민감도 (낮을수록 민감)
min_area  = 1000  # 최소 움직임 크기

print('앱 시작!  + 민감하게  - 둔감하게  q: 종료')

while True:
    ret, frame = camera.read()
    if not ret: break
    result    = frame.copy()
    gray      = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray      = cv2.GaussianBlur(gray, (21,21), 0)

    if prev_gray is None:
        prev_gray = gray
        continue

    # 프레임 차이 계산
    diff    = cv2.absdiff(prev_gray, gray)
    _, thresh = cv2.threshold(diff, threshold, 255, cv2.THRESH_BINARY)
    thresh  = cv2.dilate(thresh, None, iterations=2)  # 노이즈 제거

    # Contour 검출
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    motion_detected = False

    for c in contours:
        if cv2.contourArea(c) < min_area:
            continue
        x, y, w, h = cv2.boundingRect(c)
        cv2.rectangle(result, (x,y), (x+w,y+h), (0,0,255), 2)
        motion_detected = True

    # 상태 표시
    if motion_detected:
        result = put_korean_text(result, '움직임 감지!', (10, 10), (0,0,255))
    else:
        result = put_korean_text(result, '대기 중...', (10, 10), (0,255,0))

    result = put_korean_text(result, f'\n민감도: {100-threshold}  (+/-로 조절)',
                             (10, 10), (0,255,255), font_medium)

    # 차이 이미지와 나란히 표시
    diff_bgr = cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)
    combined = np.hstack([result, diff_bgr])
    cv2.imshow('움직임 감지 (q:종료)', combined)

    prev_gray = gray

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'): break
    elif key == 61:
        threshold = max(5, threshold - 5)
        print(f'민감도 threshold: {threshold}')
    elif key == 45:
        threshold = min(100, threshold + 5)
        print(f'민감도 threshold: {threshold}')

camera.release()
cv2.destroyAllWindows()


#### 🎯 GenAI 활용 — 움직임 감지 확장

---
**프롬프트:**
```
Python OpenCV 움직임 감지 앱에 다음 기능을 추가해줘:
1. 움직임이 감지되면 자동으로 스크린샷 저장
   (같은 움직임으로 연속 저장 방지 — 5초 쿨다운)
2. 화면 오른쪽 하단에 '저장된 캡처: N장' 표시
3. 저장 파일명: motion_HHMMSS.png 형식
```
---

In [ ]:
# 여기에 GenAI가 생성한 코드를 붙여넣으세요
# your code here

### 4-3. 출석 체크 시뮬레이터

얼굴이 일정 시간 이상 감지되면 출석으로 처리합니다.

```
얼굴 감지 시작 → 3초 이상 유지 → 출석 완료!
                 3초 미만 이탈 → 카운터 리셋
```

> 💡 실제 출입 관리 시스템의 단순화 버전이에요!

In [ ]:
import time

camera        = cv2.VideoCapture(0)
required_secs = 3      # 출석 인정 시간 (초)
attended      = False
face_start    = None   # 얼굴 첫 감지 시각
attend_log    = []     # 출석 기록

print('출석 체크 시작!  카메라 앞에 얼굴을 보여주세요.')
print(f'{required_secs}초 이상 감지되면 출석 완료!')

while True:
    ret, frame = camera.read()
    if not ret: break
    result = frame.copy()
    h, w   = frame.shape[:2]

    # 얼굴 검출
    detections = face_detector.detect(frame)
    face_found = len(detections) > 0

    if face_found:
        if face_start is None:
            face_start = time.time()
        elapsed = time.time() - face_start
        remain  = max(0, required_secs - elapsed)

        # 진행 바
        bar_w = int((elapsed / required_secs) * (w - 40))
        bar_w = min(bar_w, w - 40)
        cv2.rectangle(result, (20, h-50), (w-20, h-20), (50,50,50), -1)
        cv2.rectangle(result, (20, h-50), (20+bar_w, h-20), (0,255,0), -1)

        if elapsed >= required_secs and not attended:
            attended = True
            attend_time = time.strftime('%H:%M:%S')
            attend_log.append(attend_time)
            print(f'출석 완료! ({attend_time})')

        if attended:
            result = put_korean_text(result, '출석 완료!', (w//2-80, h//2-30), (0,255,0))
        else:
            result = put_korean_text(result, f'{remain:.1f}초 후 출석 완료',
                                     (10, 10), (0,255,255))
    else:
        face_start = None
        result = put_korean_text(result, '얼굴을 보여주세요', (10, 10),
                                     (0,255,255))

    # 출석 기록 표시
    result = put_korean_text(result, f'출석: {len(attend_log)}명',
                             (10, 45), (0,255,255), font_small)

    cv2.imshow('출석 체크 (r:리셋  q:종료)', result)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('r'):  # 리셋
        attended   = False
        face_start = None
        print('리셋')

camera.release()
cv2.destroyAllWindows()
print(f'\n총 출석: {len(attend_log)}명')
for i, t in enumerate(attend_log):
    print(f'  {i+1}번 출석: {t}')


#### 🎯 GenAI 활용 — 출석 체크 확장

---
**프롬프트 A: 다중 출석**
```
Python OpenCV 출석 체크 앱에서
같은 사람이 두 번 출석되지 않도록
한 번 출석 완료 후 10초 동안 같은 얼굴은 무시하는 쿨다운을 추가해줘.
화면에 '쿨다운: N초' 텍스트 표시.
```

**프롬프트 B: 출석부 저장**
```
출석 체크 앱 종료 시 출석 기록을
attendance_YYYYMMDD.csv 파일로 저장하는 기능을 추가해줘.
컬럼: 번호, 출석시각
pandas를 사용해줘.
```
---

In [ ]:
# 여기에 GenAI가 생성한 코드를 붙여넣으세요
# your code here

---
## 오후 — Streamlit 미니 프로젝트

아래 4가지 주제 중 하나를 골라 완성합니다.
GenAI로 코드를 받아서 실행하고, 완성되면 Streamlit Cloud에 배포합니다.

## 5. 미니 프로젝트 — Streamlit 웹 앱 만들기

오늘 배운 **OpenVINO 검출 + 이미지 처리**를 Streamlit으로 감싸서  
브라우저에서 동작하는 웹 앱으로 완성합니다.

아래 4가지 주제 중 **하나를 골라서** GenAI 프롬프트를 실행하세요.  
각 주제마다 난이도가 표시되어 있습니다.

---

| 주제 | 난이도 | 핵심 기술 |
|---|---|---|
| A. 얼굴 모자이크 앱 | 🟢 쉬움 | 얼굴 검출 + 모자이크 |
| B. 이미지 필터 비교 앱 | 🟡 보통 | 다중 필터 + 슬라이더 |
| C. 색상 분석 앱 | 🟡 보통 | HSV 마스크 + 시각화 |
| D. 얼굴 감정 스티커 앱 | 🔴 어려움 | 얼굴 검출 + 오버레이 |

> 💡 **Streamlit 기본 원리**: 웹캠 직접 접근이 제한되므로  
> **이미지 업로드 → 처리 → 결과 표시** 방식으로 만듭니다.

---
### 🟢 주제 A — 얼굴 모자이크 앱 (쉬움)

업로드한 사진에서 얼굴을 자동으로 찾아 모자이크 처리합니다.  
슬라이더로 모자이크 강도를 조절할 수 있습니다.

**완성 화면 예시**
```
[사이드바]              [메인]
모자이크 강도: 15  |  원본 이미지  |  모자이크 적용 결과
                   |              |
                   |  [다운로드 버튼]
```

---
**프롬프트:**
```
Python Streamlit과 OpenVINO로 얼굴 모자이크 웹 앱을 만들어줘.

[앱 구조]
- 제목: "얼굴 자동 모자이크 앱"
- 이미지 업로드 (jpg, png)
- 사이드바: 모자이크 강도 슬라이더 (block_size: 5~50, 기본값 15)
- 사이드바: 신뢰도 임계값 슬라이더 (0.3~0.9, 기본값 0.5)
- 메인: 원본과 모자이크 결과를 st.columns(2)로 나란히 표시
- 검출된 얼굴 수를 st.metric()으로 표시
- 결과 이미지 다운로드 버튼

[기술 조건]
- OpenVINO 모델: face-detection-adas-0001 (FP32)
  경로: ./models/intel/face-detection-adas-0001/FP32/face-detection-adas-0001.xml
- 모자이크: ROI를 축소했다가 다시 확대 (INTER_NEAREST)
- 업로드 이미지: np.frombuffer + cv2.imdecode로 읽기
- OpenCV BGR -> RGB 변환 후 st.image() 사용
- 한국어 UI
- 파일명: mosaic_app.py
```
---

In [ ]:
# 주제 A 코드를 mosaic_app.py로 저장
mosaic_code = """
# 여기에 GenAI가 생성한 코드를 붙여넣으세요
"""

with open('mosaic_app.py', 'w', encoding='utf-8') as f:
    f.write(mosaic_code)
print('mosaic_app.py 저장 완료!')
print('실행: streamlit run mosaic_app.py')

---
### 🟡 주제 B — 이미지 필터 비교 앱 (보통)

여러 필터를 동시에 적용해서 한 화면에 격자로 비교합니다.  
필터마다 파라미터를 슬라이더로 실시간 조정할 수 있습니다.

**완성 화면 예시**
```
[사이드바]            [메인 — 2x3 격자]
필터 파라미터 조정  |  원본  |  회색조  |  블러
                   |  Canny |  세피아  |  선명화
```

---
**프롬프트:**
```
Python Streamlit과 OpenCV로 이미지 다중 필터 비교 웹 앱을 만들어줘.

[앱 구조]
- 제목: "이미지 필터 비교 앱"
- 이미지 업로드 (jpg, png)
- 사이드바에 각 필터별 파라미터 슬라이더:
    - 블러: 커널 크기 (1~31, 홀수만, step=2)
    - Canny: threshold1 (10~200), threshold2 (50~400)
    - 모자이크: block_size (5~30)
- 메인: 아래 6가지 필터 결과를 st.columns(3)으로 2행 배치
    1. 원본
    2. 회색조 (Grayscale)
    3. Gaussian 블러
    4. Canny 엣지
    5. 세피아 (numpy 행렬 변환)
    6. 선명화 (Sharpening kernel)
- 각 이미지 아래에 필터 이름 캡션 표시 (st.caption)
- 사이드바에 필터 선택 multiselect로 표시할 필터 선택 가능

[기술 조건]
- 업로드 이미지: np.frombuffer + cv2.imdecode
- BGR -> RGB 변환 후 st.image()
- 한국어 UI
- 파일명: filter_compare_app.py
```
---

In [ ]:
# 주제 B 코드를 filter_compare_app.py로 저장
filter_code = """
# 여기에 GenAI가 생성한 코드를 붙여넣으세요
"""

with open('filter_compare_app.py', 'w', encoding='utf-8') as f:
    f.write(filter_code)
print('filter_compare_app.py 저장 완료!')
print('실행: streamlit run filter_compare_app.py')

---
### 🟡 주제 C — 색상 분석 앱 (보통)

이미지에서 주요 색상을 분석하고 비율을 차트로 시각화합니다.  
HSV 슬라이더로 특정 색상 영역을 직접 골라낼 수 있습니다.

**완성 화면 예시**
```
[사이드바]             [메인]
H 범위: 40 ~ 80  |  원본  |  마스크 결과
S 범위: 60 ~ 255 |
V 범위: 60 ~ 255 |  [파이차트: 색상 비율]
색상 이름: GREEN |  GREEN 픽셀: 12,400개 (23.4%)
```

---
**프롬프트:**
```
Python Streamlit과 OpenCV로 이미지 색상 분석 웹 앱을 만들어줘.

[앱 구조]
- 제목: "이미지 색상 분석 앱"
- 이미지 업로드 (jpg, png)
- 사이드바:
    - 분석할 색상 selectbox: 빨강 / 초록 / 파랑 / 노랑 / 직접 설정
    - "직접 설정" 선택 시 H/S/V 최솟값, 최댓값 슬라이더 6개 표시
    - 미리 정의된 색상은 HSV 범위가 자동 설정됨
- 메인 상단: 원본 이미지와 마스크 결과를 st.columns(2)로 나란히
- 메인 하단:
    - st.metric()으로 해당 색상 픽셀 수와 전체 대비 비율 표시
    - plotly 또는 matplotlib으로 RED/GREEN/BLUE/OTHER 비율 파이차트
- 마스크 이미지 다운로드 버튼

[기술 조건]
- 빨강은 HSV 두 구간(0~10, 160~180) 모두 처리
- 업로드 이미지: np.frombuffer + cv2.imdecode
- BGR -> RGB 변환 후 st.image()
- 한국어 UI
- 파일명: color_analysis_app.py
```
---

In [ ]:
# 주제 C 코드를 color_analysis_app.py로 저장
color_code = """
# 여기에 GenAI가 생성한 코드를 붙여넣으세요
"""

with open('color_analysis_app.py', 'w', encoding='utf-8') as f:
    f.write(color_code)
print('color_analysis_app.py 저장 완료!')
print('실행: streamlit run color_analysis_app.py')

---
### 🔴 주제 D — 얼굴 감정 스티커 앱 (어려움)

얼굴을 검출하고 그 위에 선글라스, 모자 등 PNG 스티커를 자동으로 올립니다.  
스티커는 얼굴 크기에 맞게 자동 조정됩니다.

**완성 화면 예시**
```
[사이드바]             [메인]
스티커 선택:     |  원본  |  스티커 적용 결과
🕶 선글라스      |
🎩 모자          |  검출된 얼굴: 2개
😂 웃음 효과     |  [다운로드 버튼]
```

---
**프롬프트:**
```
Python Streamlit과 OpenVINO로 얼굴 스티커 웹 앱을 만들어줘.

[앱 구조]
- 제목: "얼굴 스티커 앱"
- 이미지 업로드: 배경 사진 (jpg, png)
- 사이드바:
    - 스티커 선택 radio: 선글라스 / 모자 / 수염 / 없음
    - 스티커 위치 오프셋 슬라이더: X 이동 (-50~50), Y 이동 (-50~50)
    - 스티커 크기 비율 슬라이더: 0.5~2.0 (기본 1.0)
    - 신뢰도 임계값: 0.3~0.9
- 메인: 원본과 스티커 결과를 st.columns(2)로 나란히
- st.metric()으로 검출된 얼굴 수 표시

[스티커 처리 방법]
- 스티커는 PNG (배경 투명) 이미지 파일로 준비
  파일명: sticker_glasses.png, sticker_hat.png, sticker_beard.png
- 스티커를 얼굴 바운딩박스 너비에 맞게 cv2.resize()
- PNG 알파 채널로 배경 합성:
    alpha = sticker[:,:,3] / 255.0
    for c in range(3):
        roi[:,:,c] = sticker[:,:,c] * alpha + roi[:,:,c] * (1 - alpha)

[기술 조건]
- OpenVINO 모델: face-detection-adas-0001 (FP32)
  경로: ./models/intel/face-detection-adas-0001/FP32/face-detection-adas-0001.xml
- 업로드 이미지: np.frombuffer + cv2.imdecode
- BGR -> RGB 변환 후 st.image()
- 스티커 파일이 없으면 st.warning()으로 안내 메시지 표시
- 한국어 UI
- 파일명: sticker_app.py
```

> 💡 **스티커 이미지 준비 팁**  
> 투명 배경 PNG가 없으면 GenAI에게 추가로 요청하세요:  
> "OpenCV로 간단한 선글라스 모양 PNG를 numpy로 직접 그려서 저장하는 코드도 만들어줘"
---

In [ ]:
# 주제 D 코드를 sticker_app.py로 저장
sticker_code = """
# 여기에 GenAI가 생성한 코드를 붙여넣으세요
"""

with open('sticker_app.py', 'w', encoding='utf-8') as f:
    f.write(sticker_code)
print('sticker_app.py 저장 완료!')
print('실행: streamlit run sticker_app.py')

---
### 완성 후 — Streamlit Cloud 배포

로컬에서 잘 동작하면 어제 배운 방법으로 인터넷에 배포할 수 있습니다.

**주의: OpenVINO 모델 파일 처리**

Streamlit Cloud는 로컬 모델 파일을 읽을 수 없습니다.  
아래 프롬프트로 모델 로드 방식을 바꾸는 코드를 받으세요.

---
**프롬프트:**
```
내 Streamlit 앱이 로컬에서는 OpenVINO 모델 파일을 직접 읽는데,
Streamlit Cloud 배포 환경에서는 모델 파일이 없어서 실행이 안 돼.

다음 두 가지 방법 중 하나로 해결하는 코드를 추가해줘:

방법 1: GitHub 저장소에 모델 파일을 함께 업로드
- .gitignore에서 모델 파일 제외하는 방법 설명
- 모델 파일이 크면 Git LFS 사용 방법 안내

방법 2: 앱 실행 시 자동 다운로드
- st.cache_resource 데코레이터로 모델을 한 번만 로드
- omz_downloader 대신 urllib 또는 requests로
  공개 URL에서 .xml, .bin 파일을 다운로드하는 함수 작성
- 다운로드 중 st.spinner()로 로딩 표시

내 앱 파일명은 [선택한 파일명].py 야.
```
---

---
## 오늘 배운 내용 정리

✅ OpenVINO 모델 클래스화
✅ 실시간 얼굴 검출 + FPS 측정
✅ 검출 영역에 모자이크/블러 효과
✅ Streamlit 웹 앱으로 미니 프로젝트 완성

### 미니 프로젝트 주제별 핵심 기술

```
A. 모자이크 앱   -> 얼굴 검출 + ROI 픽셀화
B. 필터 비교 앱  -> 다중 필터 + 슬라이더 파라미터
C. 색상 분석 앱  -> HSV 마스크 + 차트 시각화
D. 스티커 앱     -> 얼굴 검출 + PNG 알파 합성
```

> 내일은 5일차! CNN으로 **폐렴 X-Ray 분류** — 딥러닝의 실제 의료 활용 🏥